# TitanAgent 実験3: 毎トークン CoT（残留型 vs 揮発型）vs 従来ターン単位 CoT on Titans (MAC)

「**1トークン（意味単位）を決めるたびに毎回短い CoT で推論する**」方式を Titans (MAC) 上で検証する実験です（Kaggle 無料枠 T4 16GB OOM 安全・3方式合計で約 25〜40 分目安）。

### 比較する 3 つの CoT アーキテクチャ
1. **`turn_cot`（従来ターン単位 CoT）**: 1ターンにつき最初に `<think> r1 | ... | r5 </think>` をまとめて出力し、その後 `<ans> y1 y2 y3 y4 y5 </ans>` を出力。思考はストリームに残留。
2. **`stream_token_cot`（毎トークン CoT・残留型）**: 出力トークン `y_k` を 1 つ決める直前に毎回 `<t> r_k </t>` を生成し、そのまま Titans MAC の 1 本道ストリームに残す。
3. **`ephemeral_token_cot`（毎トークン CoT・揮発型 / Quiet-STaR 風）**: 出力トークン `y_k` を 1 つ決める直前に `<t> r_k </t>` を生成するが、`y_k` が確定した瞬間に `<t> r_k </t>` を**揮発（破棄）**し、クリーンな `<ans> y1..y5 </ans>` だけを Titans の長期記憶ストリームに積む。

### 検証タスク（意味単位トークン × 長期記憶・上書き × 5段ルール推論）
- **語彙**: 1概念＝1トークン（約 80 語彙）。文字崩れノイズを排し、推論と記憶の本質を比較。
- **5段の依存ルール推論**: 5つの前提スロット（`時間 / 予算 / 優先 / 経験 / 目標`）を長期記憶・上書きしつつ、前の出力トークンに依存する5トークン（`負荷 → ツール → ペース → 検証 → 戦略`）を順番に推論。
- **3つのベンチマーク（全方式で同一問題・同一シード）**:
  1. `single_turn_d4`: 1ターン・distractor 4件（基礎推論力）
  2. `long_range_d16`: 1ターン・distractor 16件（未見の長距離記憶＋推論）
  3. `multi_turn_3t_d6`: **1本道ストリームで 3 ターン連続対話**（毎ターン distractor 6件＋前提上書き）。残留型のストリーム肥大化と揮発型の記憶保全の差が最も出る本命テスト。

In [ ]:
# 1. 環境確認 & ライブラリインストール
import os, sys, torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f'Environment: Colab={IN_COLAB}, Kaggle={IN_KAGGLE}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip -q install -U titans-pytorch

In [ ]:
# 2. 実験設定（Kaggle T4 16GB OOM 安全デフォルト）
from pathlib import Path

PRESET = 'tiny'              # 'micro' (~1.4M), 'tiny' (推奨・~10.1M), 'small' (~22M)
MODEL_TYPE = 'titans'        # 'titans' または比較用 'sliding_transformer'
MODES = ['turn_cot', 'stream_token_cot', 'ephemeral_token_cot']
SFT_STEPS = 350              # turn_cot / stream_token_cot の学習ステップ数
EPHEMERAL_STEP_MULT = 2.0    # ephemeral は1シーケンス1ホップ学習のためステップ倍率を設定
EVAL_EPISODES = 30           # 各ベンチマークの評価問題数
LOG_EVERY = 50
SEED = 42

if IN_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working/checkpoints/exp3_per_token_cot')
else:
    OUTPUT_DIR = Path('./checkpoints/exp3_per_token_cot')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRESETS = {
    'micro': dict(dim=128, depth=3, segment_len=16, num_persist_mem_tokens=4, num_longterm_mem_tokens=8,  seq_len=768, batch_size=4, grad_accum=2,  lr=1.5e-3),
    'tiny':  dict(dim=256, depth=4, segment_len=32, num_persist_mem_tokens=4, num_longterm_mem_tokens=16, seq_len=768, batch_size=4, grad_accum=8,  lr=8e-4),
    'small': dict(dim=384, depth=6, segment_len=32, num_persist_mem_tokens=8, num_longterm_mem_tokens=16, seq_len=768, batch_size=2, grad_accum=16, lr=5e-4),
}
cfg = PRESETS[PRESET]
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Selected preset:', PRESET, cfg)

In [ ]:
# 3. 意味単位トークナイザー & 5段ルール推論・マルチターン生成器
import json, math, random, time
from contextlib import nullcontext
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

SPECIAL_TOKENS = [
    '<|pad|>', '<|eos|>', '<|unk|>',
    '<think>', '</think>', '<t>', '</t>',
    '<ans>', '</ans>', '<turn>', '</turn>',
]

SLOT_NAMES = ['時間', '予算', '優先', '経験', '目標']
SLOT_VALUES = {
    '時間': ['極短', '短', '中', '週末'],
    '予算': ['無料', '小額', '十分'],
    '優先': ['継続', '速効', '品質', '低負荷'],
    '経験': ['初級', '中級', '上級'],
    '目標': ['副収入', '転職', '資格', '制作', '習慣'],
}
TOPICS = ['アプリ', '英語', 'ブログ', 'デザイン', '自動化', '資格学習', '発信', '研究']
ANSWER_SLOTS = ['負荷', 'ツール', 'ペース', '検証', '戦略']
ANSWER_VALUES = {
    '負荷': ['負荷:軽', '負荷:中', '負荷:重'],
    'ツール': ['ツール:無料', 'ツール:AI課金', 'ツール:教材購入'],
    'ペース': ['ペース:毎日小刻み', 'ペース:平日固定', 'ペース:週末集中', 'ペース:短期スプリント'],
    '検証': ['検証:自己チェック', '検証:公開テスト', '検証:定量計測', '検証:レビュー依頼'],
    '戦略': ['戦略:最小継続型', '戦略:一点突破型', '戦略:設計重視型', '戦略:低コスト検証型', '戦略:週末集中型', '戦略:高速反復型'],
}
DISTRACTOR_EVENTS = [
    ['メモ:', '進捗', '確認中'], ['メモ:', '少し', '休憩'],
    ['メモ:', '別件', '整理中'], ['メモ:', '課題', '見直し'],
    ['メモ:', '環境', '調整済'], ['メモ:', '優先度', '検討中'],
    ['メモ:', '資料', '調査中'], ['メモ:', '記録', '保存済'],
]
STRUCTURAL_TOKENS = [
    '設定:', '更新:', 'メモ:', '相談:', '記憶', '現在', '=>', '|',
    '進捗', '確認中', '少し', '休憩', '別件', '整理中', '課題', '見直し',
    '環境', '調整済', '優先度', '検討中', '資料', '調査中', '記録', '保存済',
]

class SemanticTokenizer:
    def __init__(self, token_to_id):
        self.token_to_id = dict(token_to_id)
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        self.pad_token_id = self.token_to_id['<|pad|>']
        self.eos_token_id = self.token_to_id['<|eos|>']
        self.unk_token_id = self.token_to_id['<|unk|>']
        self.ans_start_id = self.token_to_id['<ans>']
        self.ans_end_id = self.token_to_id['</ans>']
        self.vocab_size = len(self.token_to_id)

    @classmethod
    def build_default(cls):
        vocab = {}
        def _add(tok):
            if tok not in vocab:
                vocab[tok] = len(vocab)
        for tok in SPECIAL_TOKENS + STRUCTURAL_TOKENS + TOPICS:
            _add(tok)
        for slot in SLOT_NAMES:
            _add(slot)
            for val in SLOT_VALUES[slot]:
                _add(f'{slot}={val}')
        for ans_slot in ANSWER_SLOTS:
            for val in ANSWER_VALUES[ans_slot]:
                _add(val)
        return cls(vocab)

    def encode_tokens(self, tokens):
        return [self.token_to_id.get(t, self.unk_token_id) for t in tokens]

    def decode_tokens(self, ids, skip_pad=True):
        return [self.id_to_token.get(i, '<|unk|>') for i in ids if not (skip_pad and i == self.pad_token_id)]

def solve_rules(slots):
    t_val, b_val, p_val, e_val, g_val = slots['時間'], slots['予算'], slots['優先'], slots['経験'], slots['目標']
    if t_val == '極短' or p_val == '低負荷':
        y1 = '負荷:軽'
    elif (p_val == '速効' and t_val in ('中', '週末')) or (p_val == '品質' and t_val == '週末'):
        y1 = '負荷:重'
    else:
        y1 = '負荷:中'
    r1 = ['記憶', f'時間={t_val}', f'優先={p_val}', '=>', y1]

    if b_val == '無料':
        y2 = 'ツール:無料'
    elif (b_val == '十分' and e_val in ('中級', '上級')) or (b_val == '小額' and e_val == '上級'):
        y2 = 'ツール:AI課金'
    else:
        y2 = 'ツール:教材購入'
    r2 = ['記憶', f'予算={b_val}', f'経験={e_val}', '=>', y2]

    if t_val == '週末':
        y3 = 'ペース:週末集中'
    elif y1 == '負荷:重' and p_val == '速効':
        y3 = 'ペース:短期スプリント'
    elif y1 == '負荷:軽' or p_val == '継続':
        y3 = 'ペース:毎日小刻み'
    else:
        y3 = 'ペース:平日固定'
    r3 = ['現在', y1, f'時間={t_val}', f'優先={p_val}', '=>', y3]

    if g_val in ('副収入', '制作') and e_val != '初級':
        y4 = '検証:公開テスト'
    elif g_val in ('転職', '資格') and y2 == 'ツール:教材購入':
        y4 = '検証:定量計測'
    elif y2 == 'ツール:AI課金':
        y4 = '検証:レビュー依頼'
    else:
        y4 = '検証:自己チェック'
    r4 = ['現在', y2, f'目標={g_val}', f'経験={e_val}', '=>', y4]

    if y3 == 'ペース:短期スプリント':
        y5 = '戦略:一点突破型'
    elif y3 == 'ペース:週末集中':
        y5 = '戦略:週末集中型'
    elif y4 == '検証:公開テスト' and y1 != '負荷:軽':
        y5 = '戦略:高速反復型'
    elif p_val == '品質' and y4 in ('検証:定量計測', '検証:レビュー依頼'):
        y5 = '戦略:設計重視型'
    elif y1 == '負荷:軽':
        y5 = '戦略:最小継続型'
    else:
        y5 = '戦略:低コスト検証型'
    r5 = ['現在', y1, y3, y4, f'優先={p_val}', '=>', y5]
    return [y1, y2, y3, y4, y5], [r1, r2, r3, r4, r5]

@dataclass
class TurnSpec:
    turn_idx: int
    topic: str
    prefix_tokens: list
    slots: dict
    stale_slots: dict
    answers: list
    thoughts: list
    stale_answers: object

@dataclass
class MultiTurnEpisode:
    turns: list
    num_distractors_per_turn: int

def _make_distractor_tokens(rng, count):
    toks = []
    for _ in range(count):
        toks.extend(rng.choice(DISTRACTOR_EVENTS))
    return toks

def generate_multiturn_episode(rng, num_turns=3, num_distractors=6, updates_per_turn=1):
    slots = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
    topics = rng.sample(TOPICS, k=min(num_turns, len(TOPICS)))
    turns = []
    for t_idx in range(num_turns):
        prefix = []
        if t_idx == 0:
            prefix.append('設定:')
            for k in SLOT_NAMES:
                prefix.append(f'{k}={slots[k]}')
        d_before = num_distractors // 2
        d_after = num_distractors - d_before
        prefix.extend(_make_distractor_tokens(rng, d_before))
        stale_slots = {}
        if updates_per_turn > 0:
            for k in rng.sample(SLOT_NAMES, k=min(updates_per_turn, len(SLOT_NAMES))):
                old_val = slots[k]
                new_val = rng.choice([v for v in SLOT_VALUES[k] if v != old_val])
                stale_slots[k] = old_val
                slots[k] = new_val
                prefix.extend(['更新:', f'{k}={new_val}'])
        prefix.extend(_make_distractor_tokens(rng, d_after))
        prefix.extend(['<turn>', '相談:', topics[t_idx]])
        answers, thoughts = solve_rules(slots)
        stale_answers = None
        if stale_slots:
            sc = dict(slots); sc.update(stale_slots)
            stale_answers, _ = solve_rules(sc)
        turns.append(TurnSpec(t_idx, topics[t_idx], prefix, dict(slots), stale_slots, answers, thoughts, stale_answers))
    return MultiTurnEpisode(turns=turns, num_distractors_per_turn=num_distractors)

def format_turn_completion_tokens(turn, mode):
    if mode == 'turn_cot':
        toks = ['<think>']
        for i, r in enumerate(turn.thoughts):
            if i > 0: toks.append('|')
            toks.extend(r)
        return toks + ['</think>', '<ans>', *turn.answers, '</ans>', '</turn>']
    if mode == 'stream_token_cot':
        toks = ['<ans>']
        for r, y in zip(turn.thoughts, turn.answers):
            toks.extend(['<t>', *r, '</t>', y])
        return toks + ['</ans>', '</turn>']
    if mode == 'ephemeral_token_cot':
        return ['<ans>', *turn.answers, '</ans>', '</turn>']
    raise ValueError(mode)

tokenizer = SemanticTokenizer.build_default()
print('Semantic vocab size:', tokenizer.vocab_size)

# サンプル表示（3方式のストリーム形式の違いを確認）
sample_ep = generate_multiturn_episode(random.Random(SEED), num_turns=2, num_distractors=2, updates_per_turn=1)
t0 = sample_ep.turns[0]
print('\n--- Sample Turn 1 Prefix ---')
print(' '.join(t0.prefix_tokens))
for m in MODES:
    comp = format_turn_completion_tokens(t0, m)
    print(f'[{m:20s}] ({len(comp):2d} toks):', ' '.join(comp))

In [ ]:
# 4. モデル構築・学習・推論・評価関数
def build_model(model_type, model_cfg, device):
    if model_type == 'titans':
        from titans_pytorch import MemoryAsContextTransformer
        return MemoryAsContextTransformer(
            num_tokens=model_cfg['num_tokens'],
            dim=model_cfg['dim'],
            depth=model_cfg['depth'],
            segment_len=model_cfg['segment_len'],
            num_persist_mem_tokens=model_cfg['num_persist_mem_tokens'],
            num_longterm_mem_tokens=model_cfg['num_longterm_mem_tokens'],
        ).to(device)
    raise ValueError(f'Unsupported model_type: {model_type}')

def get_device_and_dtype():
    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability()
        if major >= 8 and torch.cuda.is_bf16_supported():
            return torch.device('cuda'), torch.bfloat16
        if major >= 7:
            return torch.device('cuda'), torch.float16
        return torch.device('cuda'), torch.float32
    return torch.device('cpu'), torch.float32

def build_sft_example(ep, mode, tokenizer, max_seq_len, rng):
    if mode in ('turn_cot', 'stream_token_cot'):
        seq_toks, mask = [], []
        for t in ep.turns:
            seq_toks.extend(t.prefix_tokens); mask.extend([0] * len(t.prefix_tokens))
            comp = format_turn_completion_tokens(t, mode)
            seq_toks.extend(comp); mask.extend([1] * len(comp))
        seq_toks.append('<|eos|>'); mask.append(1)
        ids = tokenizer.encode_tokens(seq_toks)
        labels = [i if m == 1 else -100 for i, m in zip(ids, mask)]
        return ids[:max_seq_len], labels[:max_seq_len]

    target_t_idx = rng.randrange(len(ep.turns))
    target_hop = rng.randrange(len(ANSWER_SLOTS))
    prefix_toks = []
    for idx in range(target_t_idx):
        tp = ep.turns[idx]
        prefix_toks.extend(tp.prefix_tokens)
        prefix_toks.extend(format_turn_completion_tokens(tp, 'ephemeral_token_cot'))
    tc = ep.turns[target_t_idx]
    prefix_toks.extend(tc.prefix_tokens)
    prefix_toks.append('<ans>')
    prefix_toks.extend(tc.answers[:target_hop])
    target_toks = ['<t>', *tc.thoughts[target_hop], '</t>', tc.answers[target_hop], '<|eos|>']
    p_ids = tokenizer.encode_tokens(prefix_toks)
    t_ids = tokenizer.encode_tokens(target_toks)
    ids = p_ids + t_ids
    labels = [-100] * len(p_ids) + list(t_ids)
    if len(ids) > max_seq_len:
        ov = len(ids) - max_seq_len
        ids, labels = ids[ov:], labels[ov:]
    return ids, labels

def collate_batch(items, pad_id):
    max_len = max(len(ids) for ids, _ in items)
    x = torch.full((len(items), max_len), pad_id, dtype=torch.long)
    y = torch.full((len(items), max_len), -100, dtype=torch.long)
    for i, (ids, lbls) in enumerate(items):
        x[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        y[i, :len(lbls)] = torch.tensor(lbls, dtype=torch.long)
    return x, y

def cosine_lr(step, warmup, total, max_lr, min_ratio=0.1):
    if step < warmup:
        return max_lr * (step + 1) / max(1, warmup)
    p = min(max((step - warmup) / max(1, total - warmup), 0.0), 1.0)
    return max_lr * (min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * p)))

def sample_train_episode(rng):
    num_turns = rng.choices([1, 2, 3], weights=[0.25, 0.35, 0.40])[0]
    num_dist = rng.randint(2, 10)
    updates = rng.choices([0, 1], weights=[0.20, 0.80])[0]
    return generate_multiturn_episode(rng, num_turns=num_turns, num_distractors=num_dist, updates_per_turn=updates)

def train_mode(mode, model_cfg, device, amp_dtype, steps):
    torch.manual_seed(SEED)
    rng = random.Random(SEED)
    model = build_model(MODEL_TYPE, model_cfg, device)
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    use_amp = device.type == 'cuda' and amp_dtype in (torch.bfloat16, torch.float16)
    use_scaler = device.type == 'cuda' and amp_dtype == torch.float16
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], betas=(0.9, 0.95), weight_decay=0.05)
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler) if hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler(enabled=use_scaler)
    warmup = max(1, min(40, steps // 10))
    model.train()
    t0 = time.time()
    print(f'\n=== [Train: {mode}] params={params_m:.2f}M | steps={steps} | batch={cfg["batch_size"]}x{cfg["grad_accum"]} ===')
    len_sum, final_loss = 0.0, 0.0
    for step in range(1, steps + 1):
        cur_lr = cosine_lr(step - 1, warmup, steps, cfg['lr'])
        for pg in optimizer.param_groups: pg['lr'] = cur_lr
        optimizer.zero_grad(set_to_none=True)
        accum_loss, step_lens = 0.0, []
        for _ in range(cfg['grad_accum']):
            items = [build_sft_example(sample_train_episode(rng), mode, tokenizer, cfg['seq_len'], rng) for _ in range(cfg['batch_size'])]
            step_lens.extend(len(ids) for ids, _ in items)
            x, y = collate_batch(items, tokenizer.pad_token_id)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            amp_ctx = torch.amp.autocast(device_type=device.type, dtype=amp_dtype) if use_amp else nullcontext()
            with amp_ctx:
                logits = model(x[:, :-1])
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y[:, 1:].reshape(-1), ignore_index=-100)
                scaled = loss / cfg['grad_accum']
            if use_scaler:
                scaler.scale(scaled).backward()
            else:
                scaled.backward()
            accum_loss += float(loss.item()) / cfg['grad_accum']
        if use_scaler:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        len_sum += sum(step_lens) / len(step_lens)
        final_loss = accum_loss
        if step % LOG_EVERY == 0 or step == 1 or step == steps:
            print(f'  [{mode}] step {step:4d}/{steps} | loss {accum_loss:.4f} | avg_len {sum(step_lens)/len(step_lens):.1f} | lr {cur_lr:.2e} | {time.time()-t0:.1f}s')
    torch.save({'mode': mode, 'steps': steps, 'model_state_dict': model.state_dict()}, OUTPUT_DIR / f'{mode}_latest.pt')
    if device.type == 'cuda': torch.cuda.empty_cache()
    return model, {'steps': steps, 'final_loss': round(final_loss, 4), 'avg_train_seq_len': round(len_sum / steps, 1), 'train_seconds': round(time.time() - t0, 1)}

@torch.no_grad()
def _greedy_gen(model, prefix_ids, stop_ids, max_new, max_seq_len, device):
    seq = list(prefix_ids[-(max_seq_len - 4):]) if len(prefix_ids) >= max_seq_len - 4 else list(prefix_ids)
    gen = []
    for _ in range(max_new):
        if len(seq) >= max_seq_len: break
        logits = model(torch.tensor([seq], dtype=torch.long, device=device))[:, -1, :]
        nxt = int(torch.argmax(logits, dim=-1).item())
        seq.append(nxt); gen.append(nxt)
        if nxt in stop_ids: break
    return gen

def _extract_answers(tokens):
    in_ans, in_t, preds = False, False, []
    for tok in tokens:
        if tok == '<ans>': in_ans = True; continue
        if tok == '</ans>': break
        if not in_ans: continue
        if tok == '<t>': in_t = True; continue
        if tok == '</t>': in_t = False; continue
        if not in_t: preds.append(tok)
    return preds[:len(ANSWER_SLOTS)]

@torch.no_grad()
def run_multiturn_inference(model, ep, mode, device):
    model.eval()
    stream_ids, turn_results, total_gen = [], [], 0
    end_turn_id = tokenizer.token_to_id['</turn>']
    eos_id = tokenizer.eos_token_id
    for t in ep.turns:
        stream_ids.extend(tokenizer.encode_tokens(t.prefix_tokens))
        prompt_len = len(stream_ids)
        if mode in ('turn_cot', 'stream_token_cot'):
            gen_ids = _greedy_gen(model, stream_ids, {end_turn_id, eos_id}, 80 if mode == 'turn_cot' else 90, cfg['seq_len'], device)
            total_gen += len(gen_ids)
            stream_ids.extend(gen_ids)
            gen_toks = tokenizer.decode_tokens(gen_ids)
            preds = _extract_answers(gen_toks)
            trace = ' '.join(gen_toks)
        else:
            stream_ids.append(tokenizer.ans_start_id)
            preds, hop_traces = [], []
            for _ in range(len(ANSWER_SLOTS)):
                gen_ids = _greedy_gen(model, stream_ids, {eos_id}, 20, cfg['seq_len'], device)
                total_gen += len(gen_ids)
                gen_toks = tokenizer.decode_tokens(gen_ids)
                hop_traces.append(' '.join(gen_toks))
                chosen = '<|unk|>'
                if '</t>' in gen_toks:
                    idx = gen_toks.index('</t>') + 1
                    if idx < len(gen_toks) and gen_toks[idx] != '<|eos|>':
                        chosen = gen_toks[idx]
                elif gen_toks:
                    chosen = gen_toks[0]
                preds.append(chosen)
                stream_ids.append(tokenizer.token_to_id.get(chosen, tokenizer.unk_token_id))
            stream_ids.extend([tokenizer.ans_end_id, end_turn_id])
            trace = ' || '.join(hop_traces)
        gt = t.answers
        padded = preds + [''] * max(0, len(gt) - len(preds))
        hits = [int(padded[i] == gt[i]) for i in range(len(gt))]
        stale_hit = int(t.stale_answers is not None and any(t.stale_answers[i] != gt[i] and padded[i] == t.stale_answers[i] for i in range(len(gt))))
        turn_results.append({'turn_idx': t.turn_idx, 'prompt_len': prompt_len, 'exact': int(all(hits)), 'hits': hits, 'stale_hit': stale_hit, 'preds': preds, 'gt': gt, 'trace': trace})
    return {'turns': turn_results, 'final_stream_len': len(stream_ids), 'total_gen': total_gen}

def evaluate_suite(model, mode, device):
    specs = {
        'single_turn_d4': dict(num_turns=1, num_distractors=4, updates_per_turn=1),
        'long_range_d16': dict(num_turns=1, num_distractors=16, updates_per_turn=1),
        'multi_turn_3t_d6': dict(num_turns=3, num_distractors=6, updates_per_turn=1),
    }
    results = {}
    for bname, sc in specs.items():
        rng = random.Random(SEED + 9999 + sum(ord(c) for c in bname))
        t0 = time.time()
        nt = sc['num_turns']
        t_exact, t_slot, t_y5, t_stale, t_plen = [0]*nt, [0.0]*nt, [0]*nt, [0]*nt, [0.0]*nt
        flen_sum, gen_sum, printed = 0.0, 0.0, False
        for _ in range(EVAL_EPISODES):
            ep = generate_multiturn_episode(rng, **sc)
            out = run_multiturn_inference(model, ep, mode, device)
            flen_sum += out['final_stream_len']; gen_sum += out['total_gen']
            for tr in out['turns']:
                ti = tr['turn_idx']
                t_exact[ti] += tr['exact']; t_slot[ti] += sum(tr['hits'])/len(ANSWER_SLOTS)
                t_y5[ti] += tr['hits'][-1]; t_stale[ti] += tr['stale_hit']; t_plen[ti] += tr['prompt_len']
            if not printed and bname == 'multi_turn_3t_d6':
                printed = True
                lt = out['turns'][-1]
                print(f'  [Sample {mode} | 3T Turn 3] prompt_len={lt["prompt_len"]} | exact={bool(lt["exact"])}')
                print(f'    GT:    {lt["gt"]}')
                print(f'    Pred:  {lt["preds"]}')
                print(f'    Trace: {lt["trace"][:220]}')
        n = EVAL_EPISODES
        results[bname] = {
            'overall_exact_acc': round(sum(t_exact)/(n*nt), 3),
            'overall_slot_acc': round(sum(t_slot)/(n*nt), 3),
            'overall_y5_acc': round(sum(t_y5)/(n*nt), 3),
            'overall_stale_rate': round(sum(t_stale)/(n*nt), 3),
            'avg_final_stream_len': round(flen_sum/n, 1),
            'avg_total_gen_tokens': round(gen_sum/n, 1),
            'eval_seconds': round(time.time()-t0, 1),
            'per_turn': {f'turn_{i+1}': {'exact_acc': round(t_exact[i]/n, 3), 'y5_acc': round(t_y5[i]/n, 3), 'stale_rate': round(t_stale[i]/n, 3), 'avg_prompt_len': round(t_plen[i]/n, 1)} for i in range(nt)},
        }
    return results

In [ ]:
# 5. 3方式（turn_cot / stream_token_cot / ephemeral_token_cot）の学習＆比較評価を実行
device, amp_dtype = get_device_and_dtype()
model_cfg = {
    'num_tokens': tokenizer.vocab_size,
    'dim': cfg['dim'],
    'depth': cfg['depth'],
    'segment_len': cfg['segment_len'],
    'num_persist_mem_tokens': cfg['num_persist_mem_tokens'],
    'num_longterm_mem_tokens': cfg['num_longterm_mem_tokens'],
    'seq_len': cfg['seq_len'],
}
print(f'Device={device}, amp_dtype={amp_dtype}, model_cfg={model_cfg}')

all_metrics = {'preset': PRESET, 'model_type': MODEL_TYPE, 'model_cfg': model_cfg, 'modes': {}}

for mode in MODES:
    steps = int(round(SFT_STEPS * EPHEMERAL_STEP_MULT)) if mode == 'ephemeral_token_cot' else SFT_STEPS
    model, train_stats = train_mode(mode, model_cfg, device, amp_dtype, steps)
    eval_stats = evaluate_suite(model, mode, device)
    all_metrics['modes'][mode] = {'train': train_stats, 'eval': eval_stats}
    print(f'\n[{mode}] Eval Result:')
    print(json.dumps(eval_stats, ensure_ascii=False, indent=2))
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(all_metrics, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
# 6. 最終比較サマリー表の表示
print('=== Experiment 3: Per-Token CoT Comparison Summary ===')
header = f'{"Mode":20s} | {"d4_acc":>6s} | {"d16_acc":>7s} | {"3T_acc":>6s} | {"T1":>5s} | {"T2":>5s} | {"T3":>5s} | {"3T_len":>6s} | {"3T_gen":>6s} | {"stale":>5s}'
print(header)
print('-' * len(header))
for mode, data in all_metrics['modes'].items():
    ev = data['eval']
    mt = ev['multi_turn_3t_d6']
    pt = mt['per_turn']
    print(
        f'{mode:20s} | '
        f'{ev["single_turn_d4"]["overall_exact_acc"]:6.3f} | '
        f'{ev["long_range_d16"]["overall_exact_acc"]:7.3f} | '
        f'{mt["overall_exact_acc"]:6.3f} | '
        f'{pt["turn_1"]["exact_acc"]:5.3f} | '
        f'{pt["turn_2"]["exact_acc"]:5.3f} | '
        f'{pt["turn_3"]["exact_acc"]:5.3f} | '
        f'{mt["avg_final_stream_len"]:6.1f} | '
        f'{mt["avg_total_gen_tokens"]:6.1f} | '
        f'{mt["overall_stale_rate"]:5.3f}'
    )